In [1]:
#Imports and load

In [2]:
import pandas as pd
import numpy as np
import os

RAW_DATA_PATH = "../data/raw/"
PROCESSED_DATA_PATH = "../data/processed/"

#load hourly data
df_hourly_raw = pd.read_csv(RAW_DATA_PATH + "saleshourly.csv")

#creating copy of raw data
df_hourly = df_hourly_raw.copy()

print(f"Raw shape: {df_hourly_raw.shape}")
print(f"copy shape: {df_hourly.shape}")

Raw shape: (50532, 13)
copy shape: (50532, 13)


In [3]:
#before conversion
print(f"Before: {df_hourly['datum'].dtype}")
print(f"Sample: {df_hourly['datum'].head(3)}")

#convert to datetime
df_hourly['datum'] = pd.to_datetime(df_hourly['datum'])

#after conversion
print(f"After: {df_hourly['datum'].dtype}")
print(f"Sample: {df_hourly['datum'].head(3)}")

Before: str
Sample: 0     1/2/2014 8:00
1     1/2/2014 9:00
2    1/2/2014 10:00
Name: datum, dtype: str
After: datetime64[us]
Sample: 0   2014-01-02 08:00:00
1   2014-01-02 09:00:00
2   2014-01-02 10:00:00
Name: datum, dtype: datetime64[us]


In [4]:
#Extract time features

In [5]:
import datetime as dt

df_hourly['year'] = df_hourly['datum'].dt.year
df_hourly['month'] = df_hourly['datum'].dt.month
df_hourly['day'] = df_hourly['datum'].dt.day
df_hourly['hour'] = df_hourly['datum'].dt.hour
df_hourly['day_of_week'] = df_hourly['datum'].dt.dayofweek
df_hourly['is_weekend'] = df_hourly['day_of_week'].isin([5,6]).astype(int)

print(df_hourly[['datum', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend']].head())

                datum  year  month  day  hour  day_of_week  is_weekend
0 2014-01-02 08:00:00  2014      1    2     8            3           0
1 2014-01-02 09:00:00  2014      1    2     9            3           0
2 2014-01-02 10:00:00  2014      1    2    10            3           0
3 2014-01-02 11:00:00  2014      1    2    11            3           0
4 2014-01-02 12:00:00  2014      1    2    12            3           0


In [6]:
#The Cleaning Pipeline

In [7]:
def clean_hourly_data(df_raw):

    """
    Clean the hourly sales dataset.
    
    Steps:
        1. Create working copy
        2. Convert datum to datetime
        3. Rename columns to lowercase
        4. Extract datetime features
        5. Add is_weekend flag
        6. Add is_open flag for business hours
        7. Sort by datetime
        
    Args:
        df_raw (pd.DataFrame): Raw hourly dataframe
        
    Returns:
        pd.DataFrame: Cleaned dataframe
    """
    df = df_raw.copy()

    df['datum'] = pd.to_datetime(df['datum'])

    df.columns = df.columns.str.lower().str.replace(' ','_')

    df['year'] = df['datum'].dt.year
    df['month'] = df['datum'].dt.month
    df['day'] = df['datum'].dt.day
    df['hour'] = df['datum'].dt.hour
    df['day_of_week'] = df['datum'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

    df['is_open'] = df['hour'].between(7,20).astype(int)

    df = df.sort_values('datum').reset_index(drop=True)

    print(df[['datum', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'is_open']].head())

    return df

df_hourly_clean = clean_hourly_data(df_hourly_raw)

                datum  year  month  day  hour  day_of_week  is_weekend  \
0 2014-01-02 08:00:00  2014      1    2     8            3           0   
1 2014-01-02 09:00:00  2014      1    2     9            3           0   
2 2014-01-02 10:00:00  2014      1    2    10            3           0   
3 2014-01-02 11:00:00  2014      1    2    11            3           0   
4 2014-01-02 12:00:00  2014      1    2    12            3           0   

   is_open  
0        1  
1        1  
2        1  
3        1  
4        1  


In [8]:
#VALIDATING CLEANED DATA

In [9]:
def validate_cleaned_data(df, name):

    print(f"\nValidating: {name}")
    print("="*40)
    
    passes = 0
    fails = 0

    #CHECK DATUM COLUMN DATATYPE IS DATETIME 
    if pd.api.types.is_datetime64_any_dtype(df['datum']):
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1

    #CHECK NO COLUMN NAMES HAVE SPACES
    has_spaces = any(' ' in col for col in df.columns)
    if not has_spaces:
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1

    #CHECK WEEKEND CONTAINS 0 AND 1
    if df['is_weekend'].isin([0,1]).all():
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1

    #CHECK IS_OPEN  ONLY CONTAINS 0 AND 1
    if df['is_open'].isin([0,1]).all():
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1
    

    print("-"*40)
    print(f"TOTAL PASSES : {passes}")
    print(f"TOTAL FAILS : {fails}")
    print("-"*40)

validate_cleaned_data(df_hourly_clean, "Hourly sales data")


Validating: Hourly sales data
PASS
PASS
PASS
PASS
----------------------------------------
TOTAL PASSES : 4
TOTAL FAILS : 0
----------------------------------------


In [10]:
import os

PROCESSED_DATA_PATH = "../data/processed/"

filepath = PROCESSED_DATA_PATH + "sales_hourly_clean.csv"
df_hourly_clean.to_csv(filepath, index=False)

print(f"Hourly cleaned data successfully saved at location : {filepath}")

Hourly cleaned data successfully saved at location : ../data/processed/sales_hourly_clean.csv


In [11]:
#load hourly data
df_daily_raw = pd.read_csv(RAW_DATA_PATH + "salesdaily.csv")

#creating copy of raw data
df_daily = df_daily_raw.copy()

print(f"Raw shape: {df_daily_raw.shape}")
print(f"copy shape: {df_daily.shape}")

Raw shape: (2106, 13)
copy shape: (2106, 13)


In [12]:
def clean_daily_data(df_raw):
    
    df = df_raw.copy()
    
    df['datum'] = pd.to_datetime(df['datum'])

    df.columns = df.columns.str.lower().str.replace(' ','_')

    df = df.drop(columns=['year','month','hour','weekday_name'])

    df['year'] = df['datum'].dt.year
    df['month'] = df['datum'].dt.month
    df['day'] = df['datum'].dt.day
    df['day_of_week'] = df['datum'].dt.dayofweek

    df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

    df = df.sort_values('datum').reset_index(drop=True)

    print(df[['datum', 'year', 'month', 'day', 'day_of_week', 'is_weekend']].head())

    return df

df_daily_clean = clean_daily_data(df_daily_raw)

       datum  year  month  day  day_of_week  is_weekend
0 2014-01-02  2014      1    2            3           0
1 2014-01-03  2014      1    3            4           0
2 2014-01-04  2014      1    4            5           1
3 2014-01-05  2014      1    5            6           1
4 2014-01-06  2014      1    6            0           0


In [13]:
def validate_cleaned_data(df, name):

    print(f"\nValidating: {name}")
    print("="*40)
    
    passes = 0
    fails = 0

    #CHECK DATUM COLUMN DATATYPE IS DATETIME 
    if pd.api.types.is_datetime64_any_dtype(df['datum']):
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1

    #CHECK NO COLUMN NAMES HAVE SPACES
    has_spaces = any(' ' in col for col in df.columns)
    if not has_spaces:
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1

    #CHECK WEEKEND CONTAINS 0 AND 1
    if df['is_weekend'].isin([0,1]).all():
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1

    '''
    #CHECK IS_OPEN  ONLY CONTAINS 0 AND 1
    if df['is_open'].isin([0,1]).all():
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1
    '''

    print("-"*40)
    print(f"TOTAL PASSES : {passes}")
    print(f"TOTAL FAILS : {fails}")
    print("-"*40)

validate_cleaned_data(df_daily_clean, "Daily sales data")


Validating: Daily sales data
PASS
PASS
PASS
----------------------------------------
TOTAL PASSES : 3
TOTAL FAILS : 0
----------------------------------------


In [14]:
RAW_PROCESSED_PATH = '../data/processed/'

filepath1 = RAW_PROCESSED_PATH + 'sales_daily_clean.csv'
df_daily_clean.to_csv(filepath1, index = False)

print(f"Daily cleaned data successfully saved at path: {filepath1}")

Daily cleaned data successfully saved at path: ../data/processed/sales_daily_clean.csv


In [15]:
#load weekly data
df_weekly_raw = pd.read_csv(RAW_DATA_PATH + "salesweekly.csv")

In [16]:
def clean_weekly_data(df_raw):

    df = df_raw.copy()

    df['datum'] = pd.to_datetime(df['datum'])

    df.columns = df.columns.str.lower().str.replace(' ','_')

    df['year'] = df['datum'].dt.year
    df['month'] = df['datum'].dt.month
    df['week_number'] = df['datum'].dt.isocalendar().week

    df = df.sort_values('datum').reset_index(drop=True)

    print(df[['datum', 'year', 'week_number']].head())

    return df
    
df_weekly_clean = clean_weekly_data(df_weekly_raw)

       datum  year  week_number
0 2014-01-05  2014            1
1 2014-01-12  2014            2
2 2014-01-19  2014            3
3 2014-01-26  2014            4
4 2014-02-02  2014            5


In [17]:
def validate_cleaned_data(df,name):
    print(f"\nValidating: {name}")
    print("="*40)
    
    passes = 0
    fails = 0

    #CHECK DATUM COLUMN DATATYPE IS DATETIME 
    if pd.api.types.is_datetime64_any_dtype(df['datum']):
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1

    #CHECK NO COLUMN NAMES HAVE SPACES
    has_spaces = any(' ' in col for col in df.columns)
    if not has_spaces:
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1

    '''
    #CHECK WEEKEND CONTAINS 0 AND 1
    if df['is_weekend'].isin([0,1]).all():
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1
    '''
    '''
    #CHECK IS_OPEN  ONLY CONTAINS 0 AND 1
    if df['is_open'].isin([0,1]).all():
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1
    '''

    print("-"*40)
    print(f"TOTAL PASSES : {passes}")
    print(f"TOTAL FAILS : {fails}")
    print("-"*40)

validate_cleaned_data(df_weekly_clean, "Weekly sales data")


Validating: Weekly sales data
PASS
PASS
----------------------------------------
TOTAL PASSES : 2
TOTAL FAILS : 0
----------------------------------------


In [18]:
RAW_PROCESSED_PATH = '../data/processed/'

filepath2 = RAW_PROCESSED_PATH + 'sales_weekly_clean.csv'
df_weekly_clean.to_csv(filepath2, index = False)

print(f"Weekly cleaned data successfully saved at: {filepath2}")

Weekly cleaned data successfully saved at: ../data/processed/sales_weekly_clean.csv


In [19]:
#load monthly data
df_monthly_raw = pd.read_csv(RAW_DATA_PATH + "salesmonthly.csv")

In [20]:
def clean_monthly_data(df_raw):

    df = df_raw.copy()

    df['datum'] =pd.to_datetime(df['datum'])

    df.columns = df.columns.str.lower().str.replace(' ', '_')

    df['year'] = df['datum'].dt.year
    df['month'] = df['datum'].dt.month

    df = df.sort_values('datum').reset_index(drop = True)

    print(df[['year', 'month']].head())

    return df

df_monthly_clean = clean_monthly_data(df_monthly_raw)

   year  month
0  2014      1
1  2014      2
2  2014      3
3  2014      4
4  2014      5


In [21]:
def validate_cleaned_data(df,name):
    print(f"\nValidating: {name}")
    print("="*40)
    
    passes = 0
    fails = 0

    #CHECK DATUM COLUMN DATATYPE IS DATETIME 
    if pd.api.types.is_datetime64_any_dtype(df['datum']):
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1

    #CHECK NO COLUMN NAMES HAVE SPACES
    has_spaces = any(' ' in col for col in df.columns)
    if not has_spaces:
        print("PASS")
        passes += 1
    else:
        print("FAIL")
        fails += 1

    
    #CHECK WEEKEND CONTAINS 0 AND 1
    if 'is_weekend' in df.columns:
        if df['is_weekend'].isin([0,1]).all():
            print("PASS")
            passes += 1
        else:
            print("FAIL")
            fails += 1
    
    #CHECK IS_OPEN  ONLY CONTAINS 0 AND 1
    if 'is_open' in df.columns:
        if df['is_open'].isin([0,1]).all():
            print("PASS")
            passes += 1
        else:
            print("FAIL")
            fails += 1


    print("-"*40)
    print(f"TOTAL PASSES : {passes}")
    print(f"TOTAL FAILS : {fails}")
    print("-"*40)

validate_cleaned_data(df_monthly_clean, "Monthly sales data")


Validating: Monthly sales data
PASS
PASS
----------------------------------------
TOTAL PASSES : 2
TOTAL FAILS : 0
----------------------------------------


In [22]:
RAW_PROCESSED_DATA = '../data/processed/'

filepath3 = RAW_PROCESSED_DATA + "sales_monthly_clean.csv"
df_monthly_clean.to_csv(filepath3, index = False)

print(f"Monthly clean data successfully saved at path: {filepath3}")

Monthly clean data successfully saved at path: ../data/processed/sales_monthly_clean.csv
